# Preprocessing

A lot our raw data won't work well directly with machine learning models.

In [10]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Path to the raw files
csv_files = ['../data/raw/jan_2025_ontime.csv', '../data/raw/feb_2025_ontime.csv', '../data/raw/mar_2025_ontime.csv', '../data/raw/apr_2025_ontime.csv']

# 2. Loop and Load
df_list = []
for filename in csv_files:
    print(f"Reading {filename}...")

    temp_df = pd.read_csv(filename)
    
    df_list.append(temp_df)

# 3. Concatenate (The Merge)
print("Merging dataframes...")
df = pd.concat(df_list, ignore_index=True)

print(f"Loaded {df.shape[0]} rows.")
print("Columns: \n", list(df.columns))

Reading ../data/raw/jan_2025_ontime.csv...
Reading ../data/raw/feb_2025_ontime.csv...
Reading ../data/raw/mar_2025_ontime.csv...
Reading ../data/raw/apr_2025_ontime.csv...
Merging dataframes...
Loaded 2229453 rows.
Columns: 
 ['YEAR', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'OP_UNIQUE_CARRIER', 'ORIGIN_AIRPORT_ID', 'ORIGIN', 'DEST_AIRPORT_ID', 'DEST', 'CRS_DEP_TIME', 'DEP_DELAY', 'DEP_DEL15', 'CRS_ARR_TIME', 'ARR_DELAY', 'ARR_DEL15', 'CANCELLED', 'DIVERTED', 'DISTANCE']


## Remove Cancelled and Diverted Flights

In [11]:
df_clean = df[(df['CANCELLED'] == 0) & (df['DIVERTED'] == 0)].copy()

# 5. Drop Missing Targets
df_clean = df_clean.dropna(subset=['ARR_DELAY'])

# 6. Reset Index
df_clean = df_clean.reset_index(drop=True)

print(f"Cleaned Data Shape: {df_clean.shape}")

Cleaned Data Shape: (2188776, 18)


## Make Time Cyclic

In [12]:
def to_minutes(time_val):
    """Converts HHMM (int) to Minutes since Midnight"""
    hours = time_val // 100
    minutes = time_val % 100
    return (hours * 60) + minutes

def create_cyclic_features(df, col_name, period):
    """Creates Sin/Cos features from a column"""
    # 1. Convert to Radians (0 to 2π)
    radians = (df[col_name] / period) * 2 * np.pi
    
    # 2. Calculate Sin and Cos
    sin_feat = np.sin(radians)
    cos_feat = np.cos(radians)
    return sin_feat, cos_feat

# 1. Convert HHMM to Minutes
df_clean['dep_minutes'] = to_minutes(df_clean['CRS_DEP_TIME'])
df_clean['arr_minutes'] = to_minutes(df_clean['CRS_ARR_TIME'])

# 2. Create Cyclic Features for Time (Period = 1440 minutes)
df_clean['dep_sin'], df_clean['dep_cos'] = create_cyclic_features(df_clean, 'dep_minutes', 1440)
df_clean['arr_sin'], df_clean['arr_cos'] = create_cyclic_features(df_clean, 'arr_minutes', 1440)

# 3. Create Cyclic Features for Day of Month (Period = 31 days)
# We treat the month as a circle (Start of month is close to end of previous month)
df_clean['day_sin'], df_clean['day_cos'] = create_cyclic_features(df_clean, 'DAY_OF_MONTH', 31)

print("Cyclic Features Created (Dep, Arr, Day).")
# Verify
display(df_clean[['CRS_DEP_TIME', 'dep_sin', 'dep_cos', 'DAY_OF_MONTH', 'day_sin']].head())

Cyclic Features Created (Dep, Arr, Day).


,CRS_DEP_TIME,dep_sin,dep_cos,DAY_OF_MONTH,day_sin
0,500,0.965926,0.258819,1,0.201299
1,555,0.999762,0.021815,1,0.201299
2,820,0.819152,-0.573576,1,0.201299
3,1032,0.374607,-0.927184,1,0.201299
4,1700,-0.965926,-0.258819,1,0.201299


## Categorical Encoding

In [13]:
# --- A. Airlines ---
# We convert "AA", "DL", etc. into 0, 1, 2...
airline_encoder = LabelEncoder()
df_clean['airline_idx'] = airline_encoder.fit_transform(df_clean['OP_UNIQUE_CARRIER'])
print(f"Airlines Encoded: {len(airline_encoder.classes_)} unique carriers.")

# --- B. Airports ---
# Important: Origin and Dest must share the same dictionary.
# If 'JFK' is ID 50 at Origin, it must be ID 50 at Destination.
all_airports = set(df_clean['ORIGIN']).union(set(df_clean['DEST']))
airport_encoder = LabelEncoder()
airport_encoder.fit(list(all_airports))

df_clean['origin_idx'] = airport_encoder.transform(df_clean['ORIGIN'])
df_clean['dest_idx'] = airport_encoder.transform(df_clean['DEST'])
print(f"Airports Encoded: {len(airport_encoder.classes_)} unique airports.")

# --- C. Calendar (Shift to 0-based) ---
# Month: 1-12 -> 0-11
df_clean['month_idx'] = df_clean['MONTH'] - 1
# DayOfWeek: 1-7 -> 0-6
df_clean['dayofweek_idx'] = df_clean['DAY_OF_WEEK'] - 1

print("Calendar features shifted to 0-based index.")

Airlines Encoded: 14 unique carriers.
Airports Encoded: 333 unique airports.
Calendar features shifted to 0-based index.


## Numerical Scaling

In [14]:
scaler = StandardScaler()

# Reshape because scaler expects a matrix, not a list
dist_matrix = df_clean['DISTANCE'].values.reshape(-1, 1)

# Learn the mean/std and transform
df_clean['distance_scaled'] = scaler.fit_transform(dist_matrix)

print(f"Distance Scaled. Mean: {df_clean['distance_scaled'].mean():.4f} (Should be ~0)")

Distance Scaled. Mean: 0.0000 (Should be ~0)


## Selection and Save

In [17]:
# The list of features we will feed the model
FINAL_COLS = [
    # 1. Inputs (Categorical - For Embeddings)
    'airline_idx', 'origin_idx', 'dest_idx', 'month_idx', 'dayofweek_idx',
    
    # 2. Inputs (Continuous - For Math)
    'dep_sin', 'dep_cos', 'arr_sin', 'arr_cos', 'day_sin', 'day_cos',
    'distance_scaled',
    
    # 3. Targets (The Answers)
    'ARR_DELAY',   # Regression Target (Minutes)
    'ARR_DEL15'    # Classification Target (Binary)
]

# Create the final dataframe
df_final = df_clean[FINAL_COLS].copy()

# Save
OUTPUT_PATH = '../data/processed/jantoapr_2025_processed.csv'
# Ensure directory exists
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

df_final.to_csv(OUTPUT_PATH, index=False)

print(f"SUCCESS: Processed data saved to {OUTPUT_PATH}")
print(f"Final Shape: {df_final.shape}")
print("Columns:", list(df_final.columns))

SUCCESS: Processed data saved to ../data/processed/jantoapr_2025_processed.csv
Final Shape: (2188776, 14)
Columns: ['airline_idx', 'origin_idx', 'dest_idx', 'month_idx', 'dayofweek_idx', 'dep_sin', 'dep_cos', 'arr_sin', 'arr_cos', 'day_sin', 'day_cos', 'distance_scaled', 'ARR_DELAY', 'ARR_DEL15']


In [16]:
df_final

,airline_idx,origin_idx,dest_idx,month_idx,dayofweek_idx,dep_sin,dep_cos,arr_sin,arr_cos,day_sin,day_cos,distance_scaled,ARR_DELAY,ARR_DEL15
0,0,2,85,0,2,9.659258e-01,0.258819,0.878817,-0.477159,0.201299,0.97953,-0.466595,-21.0,0.0
1,0,2,85,0,2,9.997620e-01,0.021815,0.740218,-0.672367,0.201299,0.97953,-0.466595,-17.0,0.0
2,0,2,85,0,2,8.191520e-01,-0.573576,0.199368,-0.979925,0.201299,0.97953,-0.466595,-22.0,0.0
3,0,2,85,0,2,3.746066e-01,-0.927184,-0.354291,-0.935135,0.201299,0.97953,-0.466595,-23.0,0.0
4,0,2,85,0,2,-9.659258e-01,-0.258819,-0.891007,0.453990,0.201299,0.97953,-0.466595,17.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2188771,13,324,182,3,2,9.972502e-01,-0.074108,0.819152,-0.573576,-0.201299,0.97953,-0.336449,0.0,0.0
2188772,13,324,182,3,2,1.224647e-16,-1.000000,-0.511293,-0.859406,-0.201299,0.97953,-0.336449,-5.0,0.0
2188773,13,324,182,3,2,-9.914449e-01,-0.130526,-0.918791,0.394744,-0.201299,0.97953,-0.336449,-19.0,0.0
2188774,13,329,182,3,2,9.914449e-01,-0.130526,0.382683,-0.923880,-0.201299,0.97953,0.497814,8.0,0.0
